In [ ]:
# ============================================================
# 01 — DATA PIPELINE (MIND)
# Parse -> unified schema -> temporal split -> leakage test
# Fully self-contained. Hardcoded Kaggle paths.
# ============================================================
import os, glob, datetime as dt
import numpy as np, pandas as pd
from bisect import bisect_left

# ---- hardcoded paths ----
TRAIN = "/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train"
DEV   = "/kaggle/input/datasets/wrathofgod123/mind-dev/MINDsmall_dev"
print("train exists:", os.path.exists(f"{TRAIN}/behaviors.tsv"))
print("dev exists:  ", os.path.exists(f"{DEV}/behaviors.tsv"))

NEWS = ["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH  = ["impression_id","user_id","time","history","impressions"]
def pfx(x): return f"mind:{x}"


In [ ]:
# ---- parse into UNIFIED SCHEMA: articles / impressions / history ----
def parse_split(split_dir):
    news = pd.read_csv(f"{split_dir}/news.tsv", sep="\t", header=None, names=NEWS,
                       quoting=3, usecols=["news_id","category","title","abstract"])
    news["title"] = news["title"].fillna("")
    news["abstract"] = news["abstract"].fillna("")
    articles = pd.DataFrame({
        "article_id": pfx("") + news["news_id"].astype(str),
        "title": news["title"], "abstract": news["abstract"],
        "body": "",                        # MIND-small has no body
        "category": news["category"].fillna(""),
    })
    articles["article_id"] = news["news_id"].map(pfx)

    beh = pd.read_csv(f"{split_dir}/behaviors.tsv", sep="\t", header=None, names=BEH, quoting=3)
    beh["t"] = pd.to_datetime(beh["time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

    impressions, history = [], {}
    for r in beh.itertuples():
        cands, labels = [], []
        if isinstance(r.impressions, str):
            for tok in r.impressions.split():
                p = tok.split("-")
                if len(p) == 2:
                    cands.append(pfx(p[0])); labels.append(int(p[1]))
        impressions.append({"impression_id": r.impression_id, "user_id": pfx(r.user_id),
                            "time": r.t, "candidates": cands, "labels": labels})
        if isinstance(r.history, str) and r.history:
            history[pfx(r.user_id)] = [pfx(x) for x in r.history.split()]
    return articles.drop_duplicates("article_id").reset_index(drop=True), pd.DataFrame(impressions), history

art_tr, imp_tr, hist_tr = parse_split(TRAIN)
art_dv, imp_dv, hist_dv = parse_split(DEV)
print("TRAIN: articles", len(art_tr), "impressions", len(imp_tr), "users w/ history", len(hist_tr))
print("DEV:   articles", len(art_dv), "impressions", len(imp_dv), "users w/ history", len(hist_dv))


In [ ]:
# ---- TEMPORAL SPLIT verification (never random) ----
# MIND ships pre-split by time: train (Nov 9-14) -> dev (Nov 15, strictly later).
tr_min, tr_max = imp_tr["time"].min(), imp_tr["time"].max()
dv_min, dv_max = imp_dv["time"].min(), imp_dv["time"].max()
print("train time range:", tr_min, "->", tr_max)
print("dev   time range:", dv_min, "->", dv_max)
assert dv_min > tr_max, "TEMPORAL LEAK: dev starts before train ends!"
print("\nOK: dev is strictly AFTER train (temporal holdout verified).")


In [ ]:
# ---- LEAKAGE TEST (Q9): behavioural features must use only events strictly before T ----
# We verify the point-in-time counting primitive: for an impression at T, popularity
# counts only clicks with time < T. Build a rolling click index and assert no future leak.

# gather all click events per article across train+dev
click_ev = {}
for imps in (imp_tr, imp_dv):
    for row in imps.itertuples():
        if row.time is None or pd.isna(row.time): continue
        for c, lab in zip(row.candidates, row.labels):
            if lab == 1:
                click_ev.setdefault(c, []).append(row.time)
for k in click_ev: click_ev[k].sort()

def pop_before(article, T):
    tl = click_ev.get(article)
    return bisect_left(tl, T) if tl else 0

# assertion: for a sample of impressions, popularity counted must never include
# a click at time >= T
violations = 0
checked = 0
for row in imp_tr.head(5000).itertuples():
    if row.time is None or pd.isna(row.time): continue
    for c in row.candidates:
        tl = click_ev.get(c, [])
        cnt = pop_before(c, row.time)
        # verify: none of the counted events are >= T
        future = sum(1 for t in tl[:cnt] if t >= row.time)
        violations += future
        checked += 1
print(f"checked {checked} (impression,candidate) pairs")
print(f"future-click leakage violations: {violations}")
assert violations == 0, "LEAKAGE DETECTED"
print("\nOK: no future-click leakage (Q9 assertion passed).")


In [ ]:
# ---- SUMMARY ----
print("=== MIND DATA PIPELINE SUMMARY ===")
print(f"train impressions: {len(imp_tr):,}")
print(f"dev impressions:   {len(imp_dv):,}")
print(f"unique articles:   {len(pd.concat([art_tr, art_dv]).drop_duplicates('article_id')):,}")
print(f"temporal split:    train<={imp_tr['time'].max()} | dev>={imp_dv['time'].min()}")
print(f"leakage test:      PASSED (0 violations)")
print("\nUnified schema tables built: articles / impressions / history")
print("Ready for retrieval (02, 03) and ranking (04, 05).")
